# Survey Sample Evaluation

| Metric                | Input                                                             | Direction           |
|-----------------------|-------------------------------------------------------------------|---------------------|
| **fad_vggish**        | mixture MP3 to VGGish embedding, diagonal Mahalanobis dist to ref | lower = better      |
| **fad_clap**          | mixture MP3 to CLAP embedding, diagonal Mahalanobis dist to ref   | lower = better      |
| **kad_vggish**        | mixture MP3 to VGGish kernel distance proxy to ref distribution   | lower = better      |
| **kad_clap**          | mixture MP3 to CLAP kernel distance proxy to ref distribution     | lower = better      |
| **irs**               | stem WAVs to beat interval CV per stem, averaged                  | lower = more stable |
| **cbs**               | stem WAVs to fraction of windows with simultaneous beats          | higher = better     |
| **cbd**               | stem WAVs to normalized pairwise beat timing error                | lower = tighter     |
| **cocola_both**       | stem WAVs to stem–accompaniment coherence (both modes)            | higher = better     |
| **cocola_harmonic**   | stem WAVs to stem–accompaniment coherence (harmonic only)         | higher = better     |
| **cocola_percussive** | stem WAVs to stem–accompaniment coherence (percussive only)       | higher = better     |
| **ta**                | stem WAVs → beat alignment F-measure (= Beat Alignment / BA)      | higher = better     |

In [ ]:
!pip install torchvggish

In [ ]:
import collections
import collections.abc
for _attr in [
    "MutableSequence", "Callable", "Mapping", "MutableMapping",
    "Iterable", "Iterator", "MutableSet",
]:
    if not hasattr(collections, _attr):
        setattr(collections, _attr, getattr(collections.abc, _attr))

import numpy as np
np.float   = float
np.int     = int
np.complex = complex
np.object  = object
np.bool    = bool

import torch
import torch.serialization as _ts
_real_load = _ts.load
torch.load = lambda *a, **kw: _real_load(*a, **{**kw, "weights_only": False})

In [ ]:
import os
import sys
import tempfile
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import librosa
import torch
import torchaudio
import pandas as pd
from scipy.spatial.distance import cdist
from tqdm.auto import tqdm

sys.path.insert(0, str(Path(".").resolve()))

In [ ]:
AUDIO_DIR   = Path("human-study-website/audio")     # survey MP3s
STEMS_DIR   = Path("survey_stems")                  # pre-cut stems: survey_stems/MODEL/NN/stem.wav
REF_MIX     = Path("data/slakh2100/ref_mix_10s")    # 2556 × 10s WAVs
RESULTS     = Path("results")
COCOLA_CKPT = Path("mgs_evals/ckpt/cocola.ckpt")
DEVICE      = "cpu"
SEED        = 42
N_REF       = 200
CLIP_SEC    = 10.0 # trim all audio (ref + survey) to this duration

MODELS = ["MSDM", "MSLDM", "MSG-LD", "MSG-LD-ext", "MGE-LDM", "ground_truth"]
STEMS  = ["bass", "drums", "guitar", "piano"]

RESULTS.mkdir(exist_ok=True)

def detect_stems(model: str) -> list:
    d = STEMS_DIR / model
    if not d.exists():
        return list(STEMS)
    subdirs = sorted(p for p in d.iterdir() if p.is_dir())
    if not subdirs:
        return list(STEMS)
    found = sorted(f.stem for f in subdirs[0].glob("*.wav") if f.stem != "mix")
    return found if found else list(STEMS)

def load_audio(path, sr: int) -> np.ndarray:
    audio, file_sr = librosa.load(str(path), sr=None, mono=True)
    if file_sr != sr:
        audio = librosa.resample(audio, orig_sr=file_sr, target_sr=sr)
    audio = audio[:int(CLIP_SEC * sr)]
    rms = float(np.sqrt(np.mean(audio ** 2)))
    if rms > 1e-8:
        audio = audio * (0.1 / rms)
    return audio.astype(np.float32)

## Reference File Selection

200 files drawn from `data/slakh2100/ref_mix/` with seed 42.  
The same selection is used for every metric that needs a reference distribution.

In [ ]:
rng_ref   = np.random.default_rng(SEED)
all_ref   = sorted(REF_MIX.glob("*.wav"))
sel_idx   = np.sort(rng_ref.choice(len(all_ref), N_REF, replace=False))
ref_files = [all_ref[i] for i in sel_idx]

print(f"Selected {N_REF} files  (seed={SEED})")
print("Files:")
for f in ref_files:
    print(f"  {f.name}")

## Reference Embeddings

Embed the 200 reference files with VGGish (16 kHz) and CLAP (48 kHz).  
Results are cached to `results/ref_emb_*.npy` so this cell only runs once.

In [ ]:
from mgs_evals.generation._embed import VGGishEmbedder, CLAPEmbedder

vgg_embedder  = VGGishEmbedder(device=DEVICE)
clap_embedder = CLAPEmbedder(device=DEVICE)

vgg_cache  = RESULTS / f"ref_emb_vggish_{int(CLIP_SEC)}s.npy"
clap_cache = RESULTS / f"ref_emb_clap_{int(CLIP_SEC)}s.npy"

if not vgg_cache.exists():
    print(f"Computing VGGish reference embeddings (trimmed to {CLIP_SEC}s)")
    ref_audio_16 = [load_audio(f, 16_000) for f in ref_files]
    ref_emb_vgg  = np.concatenate(
        [vgg_embedder.embed([a], 16_000) for a in ref_audio_16], axis=0
    )
    np.save(vgg_cache, ref_emb_vgg)
    print(f"  Saved {vgg_cache}  shape={ref_emb_vgg.shape}")
else:
    ref_emb_vgg = np.load(vgg_cache)
    print(f"VGGish cache loaded: shape={ref_emb_vgg.shape}")

if not clap_cache.exists():
    print(f"Computing CLAP reference embeddings (trimmed to {CLIP_SEC}s)")
    ref_audio_48 = [load_audio(f, 48_000) for f in ref_files]
    ref_emb_clap = np.stack(
        [clap_embedder.embed([a], 48_000)[0] for a in ref_audio_48], axis=0
    )
    np.save(clap_cache, ref_emb_clap)
    print(f"  Saved {clap_cache}  shape={ref_emb_clap.shape}")
else:
    ref_emb_clap = np.load(clap_cache)
    print(f"CLAP cache loaded: shape={ref_emb_clap.shape}")

ref_mean_vgg  = ref_emb_vgg.mean(axis=0)
ref_std_vgg   = ref_emb_vgg.std(axis=0).clip(1e-8)
ref_mean_clap = ref_emb_clap.mean(axis=0)
ref_std_clap  = ref_emb_clap.std(axis=0).clip(1e-8)

_rng_s = np.random.default_rng(0)
_n_vgg = min(200, len(ref_emb_vgg))
_sub_a = ref_emb_vgg[_rng_s.choice(len(ref_emb_vgg), _n_vgg, replace=False)]
_sub_b = ref_emb_vgg[_rng_s.choice(len(ref_emb_vgg), _n_vgg, replace=False)]
sigma_vgg  = float(np.median(cdist(_sub_a, _sub_b))) or 1.0
sigma_clap = float(np.median(cdist(ref_emb_clap, ref_emb_clap))) or 1.0

_n_sv = min(500, len(ref_emb_vgg))
_sv   = ref_emb_vgg[_rng_s.choice(len(ref_emb_vgg), _n_sv, replace=False)]
ref_self_k_vgg  = float(
    np.exp(-cdist(_sv, _sv, "sqeuclidean") / (2 * sigma_vgg ** 2)).mean()
)
ref_self_k_clap = float(
    np.exp(-cdist(ref_emb_clap, ref_emb_clap, "sqeuclidean") / (2 * sigma_clap ** 2)).mean()
)

print()
print(f"VGGish  sigma={sigma_vgg:.4f}  ref_self_k={ref_self_k_vgg:.4f}")
print(f"CLAP    sigma={sigma_clap:.4f}  ref_self_k={ref_self_k_clap:.4f}")

## FAD / KAD

In [ ]:
def diag_mahal(emb: np.ndarray, mean: np.ndarray, std: np.ndarray) -> float:
    return float(np.sqrt(np.mean(((emb - mean) / std) ** 2)))

def kad_proxy(
    gen_emb: np.ndarray,
    ref_embs: np.ndarray,
    sigma: float,
    ref_self_k: float,
) -> float:
    sq = np.sum((ref_embs - gen_emb) ** 2, axis=1)
    cross_k = float(np.mean(np.exp(-sq / (2 * sigma ** 2))))
    return float(1.0 - 2.0 * cross_k + ref_self_k)

fad_kad_rows = []

for model in MODELS:
    model_dir = AUDIO_DIR / model
    mp3_files = sorted(model_dir.glob("*.mp3"))
    for mp3 in tqdm(mp3_files, desc=model, leave=False):
        sid = mp3.stem

        # VGGish (16 kHz): multiple frame rows per clip -> average to one vector
        wav16  = load_audio(mp3, 16_000)
        frames = vgg_embedder.embed([wav16], 16_000)  # (n_frames, 128)
        mv     = frames.mean(axis=0)                  # (128,)

        # CLAP (48 kHz): one vector per clip
        wav48 = load_audio(mp3, 48_000)
        mc    = clap_embedder.embed([wav48], 48_000)[0]  # (512,)

        fad_kad_rows.append({
            "model":      model,
            "sample_id":  sid,
            "fad_vggish": diag_mahal(mv, ref_mean_vgg,  ref_std_vgg),
            "fad_clap":   diag_mahal(mc, ref_mean_clap, ref_std_clap),
            "kad_vggish": kad_proxy(mv, ref_emb_vgg,  sigma_vgg,  ref_self_k_vgg),
            "kad_clap":   kad_proxy(mc, ref_emb_clap, sigma_clap, ref_self_k_clap),
        })

df_fad_kad = pd.DataFrame(fad_kad_rows)
df_fad_kad

## IRS / CBS / CBD

In [ ]:
from mgs_evals import IRS, CBS, CBD

rhythm_rows = []

for model in MODELS:
    stems  = detect_stems(model)
    folder = str(STEMS_DIR / model)
    if not (STEMS_DIR / model).exists():
        print(f"[SKIP] {model}: {folder} not found")
        continue

    sample_ids = sorted(d.name for d in (STEMS_DIR / model).iterdir() if d.is_dir())
    print(f"{model}: stems={stems}  n={len(sample_ids)}")

    irs_result = IRS(stems=stems).compute(folder=folder)
    cbs_result = CBS(stems=stems, window_size=0.07, num_workers=1).compute(folder=folder)
    cbd_result = CBD(stems=stems).compute(folder=folder)

    irs_per = irs_result.get("irs_sample_results", [])
    cbs_per = cbs_result.get("cbs_track_results", [])
    cbd_per = [t.get("avg_error", float("nan")) for t in cbd_result.get("cbd_track_stats", [])]

    for i, sid in enumerate(sample_ids):
        rhythm_rows.append({
            "model":     model,
            "sample_id": sid,
            "irs": irs_per[i] if i < len(irs_per) else float("nan"),
            "cbs": cbs_per[i] if i < len(cbs_per) else float("nan"),
            "cbd": cbd_per[i] if i < len(cbd_per) else float("nan"),
        })

    print(f"  IRS={irs_result['irs']:.4f}  CBS={cbs_result['cbs_mean_beat_ratio']:.4f}  CBD={cbd_result['cbd_avg_error']:.4f}")

df_rhythm = pd.DataFrame(rhythm_rows)
df_rhythm

## COCOLA

In [ ]:
from mgs_evals.coherence import COCOLA

cocola = COCOLA(checkpoint_path=str(COCOLA_CKPT), device=DEVICE)
cocola_rows = []

for model in MODELS:
    stems  = detect_stems(model)
    folder = str(STEMS_DIR / model)
    if not (STEMS_DIR / model).exists():
        print(f"[SKIP] {model}: not found")
        continue

    sample_ids = sorted(d.name for d in (STEMS_DIR / model).iterdir() if d.is_dir())
    print(f"COCOLA {model}: stems={stems}")

    cocola_result = cocola.compute(folder=folder, stems=stems)
    per_sample = cocola_result.get("cocola_sample_scores", [])

    for i, sid in enumerate(sample_ids):
        ps = per_sample[i] if i < len(per_sample) else {}
        cocola_rows.append({
            "model":            model,
            "sample_id":        sid,
            "cocola_both":      ps.get("both",       float("nan")),
            "cocola_harmonic":  ps.get("harmonic",   float("nan")),
            "cocola_percussive":ps.get("percussive", float("nan")),
        })

    print(f"  cocola_both={cocola_result['cocola_both']:.4f}"
          f"  harmonic={cocola_result['cocola_harmonic']:.4f}"
          f"  percussive={cocola_result['cocola_percussive']:.4f}"
          f"  (random_both={cocola_result['cocola_random_both']:.4f})")

df_cocola = pd.DataFrame(cocola_rows)
df_cocola

In [ ]:
cocola_rows

## Beat Alignment

In [ ]:
from mgs_evals.coherence import BeatAlignment

ba = BeatAlignment(checkpoint_path="final0", device=DEVICE)
ta_rows = []

for model in MODELS:
    stems  = detect_stems(model)
    folder = str(STEMS_DIR / model)
    if not (STEMS_DIR / model).exists():
        print(f"[SKIP] {model}: not found")
        continue

    sample_ids = sorted(d.name for d in (STEMS_DIR / model).iterdir() if d.is_dir())
    print(f"BA {model}: stems={stems}")

    ba_result = ba.compute(folder=folder, stems=stems)
    per_sample = ba_result.get("ba_sample_fmeasure", [])

    for i, sid in enumerate(sample_ids):
        ta_rows.append({
            "model":     model,
            "sample_id": sid,
            "ba":        per_sample[i] if i < len(per_sample) else float("nan"),
        })

    print(f"  ba={ba_result['ba_fmeasure']:.4f}  skipped={ba_result['ba_n_skipped']}")

df_ta = pd.DataFrame(ta_rows)
df_ta

## Assemble & Save

In [ ]:
KEY_COLS = ["model", "sample_id"]

scores = (
    df_fad_kad
    .merge(df_rhythm, on=KEY_COLS, how="outer")
    .merge(df_cocola, on=KEY_COLS, how="outer")
    .merge(df_ta,     on=KEY_COLS, how="outer")
    .sort_values(KEY_COLS)
    .reset_index(drop=True)
)

out_csv = RESULTS / "survey_eval_scores.csv"
scores.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")
print(f"Shape: {scores.shape}  ({len(scores)} samples × {len(scores.columns)} columns)")
scores